## EDA Silver Layer
Validating the silver OBT table after transformation
Checking row counts, schema, nulls, performance conversitions, speed sanity, and also ID (Uniqueness)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum
# Read from silver layer
df = spark.read.table("marathos.silver.races_obt")
print(f"Rows: {df.count()}")

## Schema Check 
Verifying column namaes and data types.

In [0]:
# Check schema
df.printSchema()

## Null Values
Checking nulls for the columns. Should be 0 after fillna. Accept performance_seconds and performance_km nulls are accepted  they are mutually exclusive.

In [0]:
# Nulls per column
df.select([
   spark_sum(col(c).isNull().cast("int")).alias(c)
   for c in df.columns
]).display()

## Performance seconds Validation 
Checking the performance_seconds for distance events. Min should be > 0 no negative values.

In [0]:
df.select("performance_seconds").describe().display()

zero_perf = df.filter(col("performance_seconds") == 0).count()
print(f"Rows with performance_seconds = 0: {zero_perf}")

## Performance km Validation 
Checking distrubution of the performance_km for the time events. Min should be > 0.

In [0]:
df.select("performance_km").describe().display()

## Speed sanity Check 
Verifying no athlete_average_speed is over 25 km/h after my sanity silver transformation.

In [0]:
df.select("athlete_average_speed").describe().display()

above_25 = df.filter(col("athlete_average_speed") > 25.0).count()
print(f"Rows with speed > 25: {above_25}")

## ID Uniqueness Check
Verifying event_id and result_id, result_id should be unique per row as primary key for fct_results.

In [0]:
total = df.count()
unique_event_ids = df.select("event_id").distinct().count()
unique_result_ids = df.select("result_id").distinct().count()
print(f"Total rows: {total}")
print(f"Unique event_ids: {unique_event_ids}")
print(f"Unique result_ids: {unique_result_ids}")

if unique_result_ids == total:
    print("result_id is unique - as PRIMARY KEY")
else:
    print(f"Waring: {total - unique_result_ids} duplicate result_ids is not found")

## fillna Validation 
Verifying the athlete_country, athlete_gender, athlete_club, athelete_age_category have no nulls after fillna. 

In [0]:
null_check = df.filter(
    col("athlete_country").isNull() |
    col("athlete_gender").isNull() |
    col("athlete_club").isNull() |
    col("athlete_age_category").isNull()
).count()

print(f"Rows with null in fillna columns: {null_check}")
if null_check == 0:
    print("fillna OK")
else:
    print(f"WARNING: {null_check} rows still have nulls")


## Top Countries 
Sanity check - top 10 countries should match EDA from bronze results

In [0]:
df.groupBy("athlete_country").count().orderBy("count", ascending=False).limit(10).display()